# 05 — Evaluation: Reproducible Paper-Quality Runs

노트북은 실행 순서만 정의하고, 핵심 로직은 `src/evaluation.py`, `src/phase.py`, `src/experiment_plots.py`에 둔다.

**평가 항목**
- Table 1: Vanilla / Periodic / Trajectory in-distribution gait quality (`survival`, `total reward`, `reward/step`, `forward velocity`)
- Table 2: Periodic / Trajectory frequency command tracking (`freq_cmd`, `zone`, `model`, `freq_meas`, `|freq_err|`, `freq_ratio`, `PLV`, `reward/step`, `survival`)
- Figure 1: Vanilla / Periodic / Trajectory reward-per-step comparison
- Figure 2: Reward-per-step vs commanded frequency (phase-conditioned models only)
- Figure 3: Commanded vs measured gait frequency
- Figure 4: Zone-aggregated frequency error and PLV
- Raw arrays: `eval_results.npz`


## 1. Setup

In [ ]:
import os
from google.colab import drive

# 1. 구글 드라이브 강제 다시 연결
drive.mount('/content/drive', force_remount=True)

# 2. 작업 폴더 존재 여부 확인
base_path = "/content/drive/MyDrive/phase_conditioned_diffusion_policy"

if os.path.exists(base_path):
    print("폴더를 찾았습니다. 해당 위치로 이동합니다.")
    %cd {base_path}
else:
    print("폴더가 없습니다. 다시 가져와야(Clone) 합니다.")
    # 아래 주소에 본인의 깃허브 토큰을 넣어 실행하세요.
    # !git clone https://{본인_토큰}@github.com/jskim730/phase_conditioned_diffusion_policy.git {base_path}

# 프로젝트 폴더로 다시 이동
%cd /content/drive/MyDrive/phase_conditioned_diffusion_policy

Mounted at /content/drive
폴더를 찾았습니다. 해당 위치로 이동합니다.
/content/drive/MyDrive/phase_conditioned_diffusion_policy
/content/drive/MyDrive/phase_conditioned_diffusion_policy


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(PosixPath('/content/drive/MyDrive/phase_conditioned_diffusion_policy'),
 PosixPath('/content/drive/MyDrive/phase_conditioned_diffusion_policy/data'),
 PosixPath('/content/drive/MyDrive/phase_conditioned_diffusion_policy/checkpoints'),
 PosixPath('/content/drive/MyDrive/phase_conditioned_diffusion_policy/results'),
 PosixPath('/content/drive/MyDrive/phase_conditioned_diffusion_policy/figures'),
 PosixPath('/content/drive/MyDrive/phase_conditioned_diffusion_policy/videos'))

In [ ]:
!pip install -q -r "{REPO_ROOT / 'requirements.txt'}"
print('✓ requirements.txt 기반 의존성 준비 완료')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 23.2 MB/s eta 0:00:00
✓ requirements.txt 기반 의존성 준비 완료


In [ ]:
import gymnasium as gym
import torch

from dataset import load_project_data
from evaluation import (
    build_eval_results_payload,
    build_frequency_sweep_protocol,
    load_evaluation_state,
    print_frequency_sweep_summary,
    print_table1_summary,
    run_frequency_sweep_evaluation,
    write_frequency_tracking_table_markdown,
    write_table1_summary_markdown,
    run_in_distribution_evaluation,
    save_eval_results_npz,
)
from experiment_plots import (
    plot_evaluation_frequency_comparison,
    plot_table1_reward_per_step_comparison,
    plot_frequency_tracking_alignment,
    plot_zone_aggregated_tracking_metrics,
)
from configs import set_global_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')


PyTorch 2.10.0+cu128, device=cuda


## 2. 데이터 및 체크포인트 로드

In [ ]:
data = load_project_data(DATA_DIR)
set_global_seed(data['seed'], deterministic=True)


42

In [ ]:
state = load_evaluation_state(data, device=device, checkpoints_dir=CHECKPOINTS_DIR)


✓ Loaded: /content/drive/MyDrive/phase_conditioned_diffusion_policy/checkpoints/vanilla_dp_ckpt.pt  [best EMA]
  Best val: (60, 0.12391136214137077)
✓ Loaded: /content/drive/MyDrive/phase_conditioned_diffusion_policy/checkpoints/phase_periodic_ckpt.pt  [best EMA]
  Best val: (50, 0.12694708909839392)
✓ Loaded: /content/drive/MyDrive/phase_conditioned_diffusion_policy/checkpoints/phase_trajectory_ckpt.pt  [best EMA]
  Best val: (50, 0.12191000487655401)

✓ Evaluation checkpoints loaded
  Vanilla DP        : 68.23M trainable
  Periodic Phase    : 68.26M trainable
  Trajectory (ours) : 68.27M trainable


## 3. 평가 프로토콜 정의

In [ ]:
DT = 0.05
MAX_STEPS = 1000
N_SEEDS_INDIST = 50
N_SEEDS_SWEEP = 10

freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=3, ood_iqr_scale=1.5)

print(f"In-dist freqs: {freq_protocol.in_freqs.round(3).tolist()}")
print(f"OOD freqs:     {freq_protocol.ood_freqs.round(3).tolist()}")
print(f"Sweep freqs:   {freq_protocol.sweep_freqs.round(3).tolist()}")
print(f"Zones:         {freq_protocol.zone_labels.tolist()}")


In-dist freqs: [1.9249999523162842, 2.049999952316284, 2.174999952316284]
OOD freqs:     [1.5499999523162842, 2.549999952316284]
Sweep freqs:   [1.5499999523162842, 1.9249999523162842, 2.049999952316284, 2.174999952316284, 2.549999952316284]
Zones:         ['OOD-low (q25-1.5IQR)', 'in-dist (q25)', 'in-dist (q50)', 'in-dist (q75)', 'OOD-high (q75+1.5IQR)']


## 4. Table 1 — In-distribution performance

In [ ]:
# 프로젝트 필수 라이브러리 일괄 재설치
!pip install -q -r requirements.txt

# 에러 메시지가 요구한 라이브러리 확실하게 추가 설치
!pip install -q "gymnasium[mujoco]"

In [ ]:
# 모델 및 시드별 영상 촬영

import os
import sys
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

# 1. 경로 및 렌더링 세팅
PROJECT_DIR = '/content/drive/MyDrive/phase_conditioned_diffusion_policy'
os.chdir(PROJECT_DIR)
if os.path.join(PROJECT_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
os.environ['MUJOCO_GL'] = 'egl'

from evaluation import evaluate_model_at_frequency

# 2. 시드 지정
class TargetSeedWrapper(gym.Wrapper):
    def __init__(self, env, target_seed):
        super().__init__(env)
        self.target_seed = target_seed

    def reset(self, seed=None, **kwargs):
        return self.env.reset(seed=self.target_seed, **kwargs)

# 3. 촬영할 시드 리스트
best_seeds = {
    'trajectory': [274, 270, 295]
    #'vanilla': [126],
    #'periodic': [143]
}

print("영상 촬영을 시작합니다...")

# 💡 모든 영상이 모일 단일 폴더 지정
TARGET_FOLDER = "videos/seed_videos"

# 4. 이중 루프 실행
for model_key, seeds in best_seeds.items():
    for s in seeds:
        print(f"\n🎬 [{model_key}] 모델 - 실제 시드 {s}번 영상 촬영 중...")

        # 💡 파일 이름 앞에 붙일 이름표(prefix) 만들기
        video_name_prefix = f"{model_key}_seed_{s}"

        # 환경 조립
        base_env = gym.make('Ant-v5', render_mode='rgb_array')
        # 💡 폴더는 TARGET_FOLDER로 고정하고, 파일명은 name_prefix로 구분
        video_env = RecordVideo(base_env, video_folder=TARGET_FOLDER, name_prefix=video_name_prefix, disable_logger=True)
        env = TargetSeedWrapper(video_env, target_seed=s)

        # 공식 평가 함수 호출
        evaluate_model_at_frequency(
            state=state,
            model_key=model_key,
            freq_hz=2.023,
            env=env,
            data=data,
            device=device,
            n_seeds=1,
            max_steps=1000,
            dt=0.05,
            deterministic_sampling=False
        )

        env.close()
        print(f"✅ 촬영 완료. 📁 파일명: {video_name_prefix}-...mp4")

print(f"\n모든 영상 촬영이 성공적으로 끝났습니다. [{TARGET_FOLDER}] 폴더를 확인해 주세요.")

🚀 다중 영상 지정 타격(Sniper) 촬영을 시작합니다...

🎬 [trajectory] 모델 - 실제 시드 274번 영상 촬영 중...
  seed 0: survival=480 steps, total_reward=   695.2, reward/step= 1.448, x_vel= 1.026

Survival:  mean=480 ± 0 steps
Reward:    mean=695.2 ± 0.0
Reward/step: mean=1.448 ± 0.000
X velocity: mean=1.026 ± 0.000
✅ 촬영 완료! 📁 파일명: trajectory_seed_274-...mp4

🎬 [trajectory] 모델 - 실제 시드 270번 영상 촬영 중...
  seed 0: survival=1000 steps, total_reward=   192.3, reward/step= 0.192, x_vel= 0.148

Survival:  mean=1000 ± 0 steps
Reward:    mean=192.3 ± 0.0
Reward/step: mean=0.192 ± 0.000
X velocity: mean=0.148 ± 0.000
✅ 촬영 완료! 📁 파일명: trajectory_seed_270-...mp4

🎬 [trajectory] 모델 - 실제 시드 295번 영상 촬영 중...
  seed 0: survival=1000 steps, total_reward=  1888.2, reward/step= 1.888, x_vel= 1.468

Survival:  mean=1000 ± 0 steps
Reward:    mean=1888.2 ± 0.0
Reward/step: mean=1.888 ± 0.000
X velocity: mean=1.468 ± 0.000
✅ 촬영 완료! 📁 파일명: trajectory_seed_295-...mp4

🎉 모든 영상 촬영이 성공적으로 끝났습니다! [videos/all_best_seeds] 폴더를 확인해 주세요.


In [ ]:
# 계산 시드 구간 조

import os
import sys
import gymnasium as gym

# 1. 경로 및 환경 복구
PROJECT_DIR = '/content/drive/MyDrive/phase_conditioned_diffusion_policy'
os.chdir(PROJECT_DIR)
if os.path.join(PROJECT_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
os.environ['MUJOCO_GL'] = 'egl'

from evaluation import evaluate_model_at_frequency

# 🎯 시드 번호를 가로채서 offset 만큼 더해준다_(수정 필요 1번)
class SeedOffsetSniper(gym.Wrapper):
    def __init__(self, env, offset=300):
        super().__init__(env)
        self.offset = offset
    def reset(self, seed=None, **kwargs):
        actual_seed = seed + self.offset if seed is not None else self.offset
        return self.env.reset(seed=actual_seed, **kwargs)

# 2. 탐색 설정
START_OFFSET = 300
SEARCH_COUNT = 50  # 300번부터 349번까지 50개 확인

print(f"🚀 시드 {START_OFFSET}번부터 {START_OFFSET + SEARCH_COUNT - 1}번까지 'trajectory' 모델 정밀 탐색 시작...")

# 3. 환경 생성
env = gym.make('Ant-v5', render_mode='rgb_array')
env = SeedOffsetSniper(env, offset=START_OFFSET)

# 4. 모델 개별 평가 함수 실행
results = evaluate_model_at_frequency(
    state=state,
    model_key='trajectory', # 여기서 모델을 하나로 고정합니다.
    freq_hz=2.023,
    env=env,
    data=data,
    device=device,
    n_seeds=SEARCH_COUNT,
    max_steps=1000,
    dt=0.05,
    deterministic_sampling=False
)

env.close()
print(f"\n✨ 시드 {START_OFFSET}번대 탐색이 완료되었습니다.")

🚀 시드 300번부터 349번까지 'trajectory' 모델 정밀 탐색 시작...


KeyboardInterrupt: 

In [ ]:
env = gym.make('Ant-v5')
table1_results = run_in_distribution_evaluation(
    state,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_INDIST,
    max_steps=MAX_STEPS,
    dt=DT,
)


=== Table 1: In-dist @ f=2.023 Hz, n=1000 seeds ===


--- Vanilla DP ---
  seed 0: survival=1000 steps, total_reward=   681.1, reward/step= 0.681, x_vel= 0.048
  seed 1: survival=1000 steps, total_reward=   694.0, reward/step= 0.694, x_vel= 0.057
  seed 2: survival=1000 steps, total_reward=   428.3, reward/step= 0.428, x_vel=-0.056
  seed 3: survival=1000 steps, total_reward=  1075.6, reward/step= 1.076, x_vel= 0.473
  seed 4: survival=1000 steps, total_reward=   679.1, reward/step= 0.679, x_vel= 0.109
  seed 5: survival=1000 steps, total_reward=   813.0, reward/step= 0.813, x_vel= 0.190
  seed 6: survival=1000 steps, total_reward=   764.8, reward/step= 0.765, x_vel= 0.128
  seed 7: survival=1000 steps, total_reward=   784.8, reward/step= 0.785, x_vel= 0.150
  seed 8: survival=1000 steps, total_reward=   825.9, reward/step= 0.826, x_vel= 0.207
  seed 9: survival=1000 steps, total_reward=   477.9, reward/step= 0.478, x_vel=-0.119
  seed 10: survival=1000 steps, total_reward=   906.2, re

In [ ]:
print_table1_summary(state, table1_results, freq_hz=float(data['freq_window_mean']))
write_table1_summary_markdown(
    state,
    table1_results,
    RESULTS_DIR / 'table1_indist_quality.md',
    freq_hz=float(data['freq_window_mean']),
    interval='ci95',
)


## 5. Table 2 — Frequency command tracking

In [ ]:
freq_results = run_frequency_sweep_evaluation(
    state,
    freq_protocol,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_SWEEP,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_frequency_sweep_summary(data, freq_protocol, freq_results)
write_frequency_tracking_table_markdown(
    freq_protocol,
    freq_results,
    RESULTS_DIR / 'table2_frequency_tracking.md',
    interval='ci95',
)


## 6. Figures — Contribution-focused visualizations


In [ ]:
plot_table1_reward_per_step_comparison(
    table1_results,
    FIGURES_DIR / 'eval_figure1_reward_per_step_comparison.png',
    interval='ci95',
)

plot_evaluation_frequency_comparison(
    table1_results,
    freq_results,
    data,
    FIGURES_DIR / 'eval_figure2_reward_per_step_vs_freq.png',
    n_seeds_sweep=N_SEEDS_SWEEP,
)

plot_frequency_tracking_alignment(
    freq_results,
    data,
    FIGURES_DIR / 'eval_figure3_target_vs_measured_freq.png',
    n_seeds_sweep=N_SEEDS_SWEEP,
)

plot_zone_aggregated_tracking_metrics(
    freq_protocol,
    freq_results,
    FIGURES_DIR / 'eval_figure4_zone_tracking_metrics.png',
)


## 7. 결과 저장 — `eval_results.npz`

In [ ]:
eval_payload = build_eval_results_payload(
    data,
    table1_results,
    freq_protocol,
    freq_results,
    n_seeds_indist=N_SEEDS_INDIST,
    n_seeds_sweep=N_SEEDS_SWEEP,
)
save_eval_results_npz(eval_payload, RESULTS_DIR / 'eval_results.npz')


## 8. 완료 체크

- 세 모델 동일 protocol로 측정
- Frequency command-tracking metric을 저장
- Notebook은 orchestration만 담당하고 재사용 로직은 `src`에 위치
